In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"


import matplotlib.pyplot as plt
import numpy as np

from openpi.policies import policy_config as _policy_config
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

In [ ]:
config = _config.get_config("pi05_libero_ada")
checkpoint_dir = "../checkpoints/pi05_libero/pi05_libero_bl/29999"

In [ ]:
# 2. Create the specific DataConfig from the factory defined in your main config.
#    This step prepares all dataset-related settings, including transforms.
data_config = config.data.create(config.assets_dirs, config.model)

# 3. Create the base dataset. This function will use the `repo_id` from
#    the data_config to load the LeRobot dataset from the hub or a local path.
base_dataset = _data_loader.create_torch_dataset(data_config, config.model.action_horizon, config.model)

# 4. Wrap the base dataset with the transforms defined in your config.
#    This applies the repack, data, and model transforms.
dataset = _data_loader.TransformedDataset(
    base_dataset,
    [
        *data_config.repack_transforms.inputs,
        # *data_config.data_transforms.inputs, # test data feed to model
        # *data_config.model_transforms.inputs, # test data feed to model
    ],
)

In [ ]:
n_steps = 10
policy = _policy_config.create_trained_policy(config, checkpoint_dir, sample_kwargs={"num_steps": n_steps})

In [ ]:
from tqdm import tqdm

max_inferences = 200
all_gt_actions = []
all_infer_actions = []
all_x_ts = []
all_v_ts = []
all_targets = []
delay = 0

for i in tqdm(range(max_inferences)):
    data = dataset[i * config.model.action_horizon]  # i * chunk_size
    if i >= max_inferences:
        break

    gt_actions = data["actions"].numpy().copy()  # Shape: (50, 14)
    # del data["actions"]

    if delay > 0:
        data["action_prefix"] = gt_actions[:delay].copy()
        data["delay"] = np.array(delay)

    data["debug"] = True
    outputs = policy.infer(data)
    infer_actions = outputs["actions"]  # Shape: (50, 14)

    all_infer_actions.append(infer_actions)
    all_gt_actions.append(gt_actions)
    all_x_ts.append(outputs["all_x_t"])
    all_v_ts.append(outputs["all_v_t"])
    all_targets.append(outputs["targets"])

In [ ]:
action_start_idx = 0
action_end_idx = 7

all_x_ts = np.stack(all_x_ts)
x_t = all_x_ts[:, :, 0, delay:, action_start_idx:action_end_idx]
print(x_t.shape)

all_v_ts = np.stack(all_v_ts)
v_t = all_v_ts[:, :, 0, delay:, action_start_idx:action_end_idx]
print(v_t.shape)

# np.save(f"x_t_s{n_steps}.npy", x_t)

all_targets = np.stack(all_targets)
targets = all_targets[:, :, delay:, action_start_idx:action_end_idx]
print(targets.shape)

In [ ]:
exp_name = checkpoint_dir.split("/")[-2]
np.save(f"../data/study/{exp_name}_x_t.npy", x_t)

## Diff analysis

In [ ]:
def metric_diff(x, m):
    if m == "l2":
        return np.linalg.norm(np.diff(x, axis=1), axis=-1)
    elif m == "rel_l2":
        return np.linalg.norm(np.diff(x, axis=1), axis=-1) / (np.linalg.norm(x[:, :-1], axis=-1) + 1e-8)
    elif m == "l1":
        return np.sum(np.abs(np.diff(x, axis=1)), axis=-1)
    elif m == "rel_l1":
        return np.sum(np.abs(np.diff(x, axis=1)), axis=-1) / (np.sum(np.abs(x[:, :-1]), axis=-1) + 1e-8)
    else:
        raise ValueError(f"Unknown metric: {m}")


x_t_diff = metric_diff(x_t, "l2")
print(x_t_diff.shape)  # (n_rollouts, n_steps, horizon)

x_t_diff_avg = np.mean(x_t_diff, axis=(0))
print(x_t_diff_avg.shape)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
im = ax.imshow(x_t_diff_avg, aspect="auto")
ax.set_yticks(np.arange(x_t_diff_avg.shape[0]))
ax.set_yticklabels(np.arange(1, x_t_diff_avg.shape[0] + 1))
ax.set_xlabel(r"$$", fontsize=14)
ax.set_ylabel(r"$$", fontsize=14)
ax.set_title(r"$$", fontsize=16)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Final analysis

In [ ]:
def metric_final(x, final, m):
    if m == "l2":
        return np.linalg.norm(x[:, 1:] - final, axis=-1)
    elif m == "l1":
        return np.sum(np.abs(x[:, 1:] - final), axis=-1)
    elif m == "mse":
        # Normalized MSE to Final Prediction
        # $$E_{t, i} = \frac{\| x_{t, i} - x_{final, i} \|_2}{\| x_{init, i} - x_{final, i} \|_2}$$
        return np.sum((x[:, 1:] - final) ** 2, axis=-1) / np.sum((x[:, 0:1] - final) ** 2, axis=-1)
    else:
        raise ValueError(f"Unknown metric: {m}")


# x_t_final = x_t[:, -1:]
x_t_final = targets
diff_final = metric_final(x_t, x_t_final, "mse")
print(diff_final.shape)

diff_final_avg = np.mean(diff_final, axis=(0))

# plt.imshow(diff_final_avg)
# for h in [0, 10, 20, 30]:
#     plt.plot(diff_final_avg[:, h], label=f"h={h}")
# plt.legend()
for t in range(n_steps):
    plt.plot(diff_final_avg[t], label=f"t={t}")
plt.legend()
plt.show()

## Estimated x1 analysis

In [ ]:
# constant time schedule
times = np.linspace(1, 0, n_steps + 1)[:-1]
dt = -1 / n_steps
# estimate x_1 from x_t and the time schedule
x_1_hat = x_t[:, :-1] - times.reshape(1, -1, 1, 1) * (np.diff(x_t, axis=1)) / dt

x_t_final = x_t[:, -1:]
diff_x1_hat = np.linalg.norm(x_1_hat - x_t_final, axis=-1)
diff_x1_hat_avg = np.mean(diff_x1_hat, axis=(0))

print(diff_x1_hat_avg.shape)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
n_rows, n_cols = diff_x1_hat_avg.shape
im = ax.imshow(diff_x1_hat_avg, aspect="auto", extent=[0, n_cols, n_rows - 0.5, -0.5])
ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels(np.arange(1, n_rows + 1))
ax.set_xlabel("action index", fontsize=16)
ax.set_ylabel(r"sampling step", fontsize=16)
ax.set_title(
    r"Deviation of estimated clean actions from final output $\|\tilde{\mathbf{A}}_{t}^{\tau\rightarrow 0}- \mathbf{A}_{t}^{0}\|$",
    fontsize=16,
)
im.set_clim(vmax=0.09)
fig.colorbar(im, ax=ax, shrink=1)
plt.tight_layout()
plt.show()

## Straightness analysis

In [ ]:
# def straightness(x):
#     # $$S_i = \frac{\sum_{t=0}^{K-1} \| x_{t+1, i} - x_{t, i} \|_2}{\| x_{final, i} - x_{init, i} \|_2}$$
#     return np.sum(np.linalg.norm(np.diff(x, axis=1), axis=-1), axis=1) / np.linalg.norm(x[:, 0] - x[:, -1], axis=-1)


def straightness(x):
    d = x[:, :1] - x[:, -1:]
    s = np.linalg.norm(d - np.diff(x, axis=1) / dt, axis=-1)
    return np.mean(s, axis=1)


stra = straightness(x_t)
stra_avg = np.mean(stra, axis=0)
stra_std = np.std(stra, axis=0)

x = np.arange(len(stra_avg))
plt.figure(figsize=(8, 6))
plt.plot(x, stra_avg, label="mean")
plt.fill_between(x, stra_avg - stra_std, stra_avg + stra_std, alpha=0.3)
plt.xlabel("action index", fontsize=16)
plt.ylabel(r"$S\ (\mathbf{A}_{t})$", fontsize=16)
plt.title("Straightness of sampling path", fontsize=16)
plt.tight_layout()
plt.show()

## Openloop plot

In [ ]:
# --- Prepare data for plotting ---
# Concatenate lists of arrays into single large arrays
if all_gt_actions:
    gt_actions_continuous = np.concatenate(all_gt_actions[:20], axis=0)
    inferred_actions_continuous = np.concatenate(all_infer_actions[:20], axis=0)

    total_steps, num_dims = gt_actions_continuous.shape
    time_steps_per_inference = gt_actions_continuous.shape[0] // 20

    # --- Plotting ---
    fig, axes = plt.subplots(7, 2, figsize=(20, 28), sharex=True)
    axes = axes.flatten()

    x_axis = np.arange(total_steps)

    for dim_idx in range(num_dims):
        ax = axes[dim_idx]

        # Plot the continuous action sequences
        ax.plot(x_axis, gt_actions_continuous[:, dim_idx], label="Ground Truth", color="cornflowerblue", alpha=0.9)
        ax.plot(
            x_axis, inferred_actions_continuous[:, dim_idx], label="Inferred", color="tomato", linestyle="--", alpha=0.9
        )

        # Mark the starting point of each inference sequence
        start_indices = np.arange(0, total_steps, time_steps_per_inference)
        ax.scatter(
            start_indices,
            gt_actions_continuous[start_indices, dim_idx],
            c="blue",
            marker="o",
            s=40,
            zorder=5,
            label="GT Start",
        )
        ax.scatter(
            start_indices,
            inferred_actions_continuous[start_indices, dim_idx],
            c="darkred",
            marker="x",
            s=40,
            zorder=5,
            label="Inferred Start",
        )

        ax.set_title(f"Action Dimension {dim_idx}")
        ax.set_ylabel("Value")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend()

    # Set common X-axis label
    fig.supxlabel(f"Continuous Timestep (across {max_inferences} inferences)")

    plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust layout to make space for suptitle
    # fig.suptitle(f'Comparison of Ground Truth and Inferred Actions @Step {steps}', fontsize=18)
    # plt.savefig(f'{ckpt_root}/inferred_vs_gt_actions-{steps}.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No data was collected for plotting.")